# Board-Level Power Measurement (KV260)

Measures dynamic power using the onboard INA260 sensor.
Run this notebook on the KV260 board with PYNQ v3.0.

In [ ]:
from snn_driver_v3 import SNNOverlay
import numpy as np
import time, threading, sys, io

POWER_PATH = '/sys/class/hwmon/hwmon2/power1_input'

def read_power_w():
    return int(open(POWER_PATH).read().strip()) / 1_000_000

def sample_power(duration_s, interval=0.2):
    samples = []
    t0 = time.time()
    while time.time() - t0 < duration_s:
        samples.append(read_power_w())
        time.sleep(interval)
    return samples

snn = SNNOverlay("snn_kv260.bit")
snn.load_weights("network_weights.bin")
image = np.fromfile("test_input_00.bin", dtype=np.int8).reshape(256, 256, 3)
print(f"Ready. Image shape: {image.shape}")

In [ ]:
print("Measuring IDLE power (10s)...")
time.sleep(2)
idle_samples = sample_power(10)
idle_avg = sum(idle_samples) / len(idle_samples)
idle_min = min(idle_samples)
idle_max = max(idle_samples)

print(f"Samples: {len(idle_samples)}")
print(f"Avg: {idle_avg:.3f} W")
print(f"Min: {idle_min:.3f} W")
print(f"Max: {idle_max:.3f} W")

In [ ]:
_real = sys.stdout
_captured = io.StringIO()
sys.stdout = _captured  

result = {}
def run_inf():
    result['logits'], result['pred'] = snn.predict(image)

inf_t = threading.Thread(target=run_inf)
inf_t.start()

time.sleep(2)
active_samples = sample_power(3)
active_avg = sum(active_samples) / len(active_samples)
dynamic = active_avg - idle_avg

_real.write(f"{'='*45}\n")
_real.write(f" ACTIVE power:   {active_avg:.3f} W\n")
_real.write(f" IDLE power:     {idle_avg:.3f} W\n")
_real.write(f" DYNAMIC power:  {dynamic:.3f} W\n")
_real.write(f" Vivado est:     0.860 W\n")
_real.write(f"{'='*45}\n")
_real.flush()